# CS 195: Natural Language Processing
## Chat and Instruct Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F2_1_ChatInstruct.ipynb)

## References


* [Hugging Face Chat Basics](https://huggingface.co/docs/transformers/en/conversations)
* [SmolLM2: When Smol Goes Big — Data-Centric Training of a Small Language Model](https://arxiv.org/pdf/2502.02737)


## Demo Day

Sit with the same people you sat with last week

* Each person do a 5-min demo of **creative synthesis** project or completed **applied exploration** (or **core practice** if that's what you have)
* Write down the names of the people you presented to (you'll include this in your portfolio later)
* (optional) Nominate a cool project to show off to everyone



## Install Modules

We'll be using `transformers` version 5. You probably only need to run this if you are doing this for the first time on your own computer. If so, uncomment these two lines and run it.


In [ ]:
#import sys
#!{sys.executable} -m pip install transformers accelerate

## Large vs. Small language models

Large Language Models (GPT 5.2, Claude Opus 4.6, Grok 4.1, Gemini 3) get all the attention.

Smaller language models have come a long way too, and they require much less computation
* can often be run on a laptop or a Colab instance
* can be *fine-tuned* for specific applications with good performance


### Example: SmolLM2

Hugging Face developed a [family of small language models called SmolLM](https://huggingface.co/collections/HuggingFaceTB/smollm2) there's also a [SmolLM3](https://huggingface.co/blog/smollm3)

SmolLM2 comes in various sizes
* 135M (135 million parameters - weights in the neural network)
* 360M
* 1.7B

Contrast with the LLMs above which likely all have over 100 billion parameters and run on a cluster of devices in a data center

Each size has a **base model**, like [SmolLM2-360M](https://huggingface.co/HuggingFaceTB/SmolLM2-360M)
* *pre-trained* on lots of diverse text
* designed to predict the *next word* - it's your phone's keyboard text prediction on steroids

The *base model* is then fine-tuned on *instruction following* and *conversational data*, which make it useful for building **chat bots**.
* resulting model has a name like [SmolLM2-360M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct)

## Building a chat bot with the `text-generation` pipeline

Setting up a chat bot works the same way as other Hugging Face models we've seen, but we'll use the `text-generation` pipeline

In [ ]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device

chatbot = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M-Instruct", device = device)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

In [ ]:
test_question1 = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Generate a basic blank html with the heading welcome."},
    ]

response = chatbot(test_question1)
print("ASSISTANT:", response[0]['generated_text'][-1]['content'])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASSISTANT: Welcome

```html
<!DOCTYPE html>
<html>
<head>
	<title>Welcome</title>
	<style>
		h1 {
			color: blue;
			text-align: center;
			font-size: 36px;
		}
	</style>
</head>
<body>
	<h1>Welcome</h1>
</body>
</html>
```

Alternatively, here is a basic HTML with the heading Welcome and some basic styling:

```html
<!DOCTYPE html>
<html>
<head>
	<title>Welcome</title>
</head>
<body>
	<h1>Welcome</h1>
</body>
</html>
```


### Chat template

*Instruct* models often allow you to pass the input in using a **Chat Template** like this

In [ ]:
chat_history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain gravity in one paragraph."},
]

Now lets get the response and display what is returned

In [ ]:
response = chatbot(chat_history)
response

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'system',
    'content': 'You are a helpful assistant.'},
   {'role': 'user', 'content': 'Explain gravity in one paragraph.'},
   {'role': 'assistant',
    'content': "Gravity is a fundamental force of nature that attracts two masses toward each other. It is the reason why objects fall towards the ground when dropped, why planets orbit around stars, and why the moon is in orbit around the Earth. Gravity is a result of the curvature of spacetime caused by the presence of mass or energy, which affects the motion of other objects. According to Einstein's theory of general relativity, gravity is not a force, but rather the result of the geometry of spacetime being curved by massive objects, such as planets. This curvature affects not only objects with mass but also light, which follows the shortest path possible in curved spacetime, a phenomenon known as the curvature effect on light."}]}]

**Notice:**
* it just uses normal lists and dictionaries
* it returns the entire chat history on the `'generated_text'` key

If you wanted to just print out the response, you could do it like this

In [ ]:
print("ASSISTANT:", response[0]['generated_text'][-1]['content'])

ASSISTANT: Gravity is a fundamental force of nature that attracts two masses toward each other. It is the reason why objects fall towards the ground when dropped, why planets orbit around stars, and why the moon is in orbit around the Earth. Gravity is a result of the curvature of spacetime caused by the presence of mass or energy, which affects the motion of other objects. According to Einstein's theory of general relativity, gravity is not a force, but rather the result of the geometry of spacetime being curved by massive objects, such as planets. This curvature affects not only objects with mass but also light, which follows the shortest path possible in curved spacetime, a phenomenon known as the curvature effect on light.


## Multi-turn conversations

If you want to have a multi-turn conversation, you need to pass *the entire chat history* to the model - it doesn't have any inherent memory of the output it just gave you

In [ ]:
another_chat_history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain gravity in one paragraph."},
    {"role": "assistant", "content": "Gravity is a fundamental force of nature that causes objects with mass to attract each other. According to Sir Isaac Newton, the force of gravity between two objects depends on their masses and the distance between them. The larger the mass of the objects, the greater the gravitational pull. This force is responsible for keeping planets in orbit around the sun and causes objects to fall towards the ground when dropped. Albert Einstein also described gravity as a curvature of spacetime caused by massive objects, which in turn warps the fabric of spacetime around them."},
    {"role": "user", "content": "Which of those two do you think has had a bigger impact on the field?"}
]

next_response = chatbot(another_chat_history)
print(next_response[0]['generated_text'][-1]['content'])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


According to Sir Isaac Newton's law of universal gravitation, gravity is the primary force at work between objects, particularly in the gravitational attraction between planets and celestial bodies. This concept laid the foundation for understanding gravity and its relationship to motion, and it has had a significant impact on various fields, including astronomy and physics.


## Exercise

Write a loop that allows for back-and-forth conversation with the model. Make sure to keep track of the full history of the chat as you go.

In [ ]:
second_chat_history = [
    {"role": "system", "content": "You are a professor."},
    {"role": "user", "content": "how has computers and technology changed over time?"},
    {"role": "assistant", "content": "Over the years, computers and technology have undergone tremendous transformations. In the early days of computers, they were massive, cumbersome devices that were primarily used for scientific and mathematical calculations. They were expensive and complex to use, making them inaccessible to most people.As computers became more accessible, the technology improved, and they were used more widely for general purpose computing. The first personal computers were released, like the IBM PC, and were affordable and user-friendly, making them more accessible to the general public.The 1980s and 1990s saw the rise of the internet, which revolutionized the way people accessed information and communicated. The internet enabled computers to become more connected, allowing for instant communication and collaboration. This led to the development of personal networks and the creation of the World Wide Web, which greatly increased the speed and accessibility of information.In recent years, we've seen significant advancements in artificial intelligence, robotics, and the Internet of Things (IoT). These technologies have transformed industries, from healthcare to manufacturing and logistics. For example, AI has been used to develop medical equipment that can diagnose diseases more accurately, while robotics has been used to automate assembly lines in factories."},
    {"role": "user", "content": "what questions would you ask to test a students knowledge on the hisotry above?"}
    ]

In [ ]:
response = chatbot(second_chat_history)
print("ASSISTANT:", response[0]['generated_text'][-1]['content'])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASSISTANT: To test a student's knowledge on the history of computers and technology, I would ask a variety of questions that cover different aspects of the topic. Here are some examples:

**General knowledge questions:**

1. What is the earliest known computer, and what was its purpose?
2. What was the first personal computer, and how did it work?
3. What is the difference between a computer and a smartphone?
4. How has technology impacted society throughout history?

**Historical context questions:**

1. What was the first computer program, and what was its purpose?
2. How did the development of computers change the way people worked and communicated?
3. What were some of the major technological advancements in computing during the 20th century?
4. How has the history of computers and technology influenced modern society?

**Technical questions:**

1. What is the difference between a CPU, motherboard, and RAM?
2. How does a computer's architecture support different types of computing 

## Evaluating Chat Models: Benchmarks

A benchmark is a dataset with one or more reference answers that can be used to measure a model's response (like the reference summaries we compared against with ROUGE)


Model benchmarking is a big deal - companies like to report how well their models perform on all kinds of benchmarks

For example, see the **performance** tab here: https://deepmind.google/models/gemini/pro/

Take a look at this benchmark for multi-turn conversations: https://huggingface.co/datasets/HuggingFaceH4/mt_bench_prompts

**Group Discussion:** What are some things you notice about this data?



## Evaluating Chat Models: Human Evaluators

Language models are often evaluated by having humans perform A/B testing where two models respond to the same prompt, and the human indicates which was better.

Try it out here: https://arena.ai

## Group Exercise

Do the following as a group:
* Come up with 5 language model prompts - what are some questions/instructions you think would help you decide how good a language model is?
* Test them using [SmolLM2-360M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct) and [Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
* Have each person in your group vote on which one they thought was the best
* Write down the results

In [ ]:
test_question1 = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Generate a basic blank html with the heading welcome."},
    ]

response = chatbot(test_question1)
print("ASSISTANT:", response[0]['generated_text'][-1]['content'])

print("=================================================================================")

test_question2 = [
    {"role": "system", "content": "You are a helpful assisant."},
    {"role": "user", "content": "If a mild steel I-beam is ten inches thick and twenty feet long, and supported rigidly on each end, about how much weight could be placed on its center point before it exhibited inelastic strain?"},
    ]

response = chatbot(test_question2)
print("ASSISTANT:", response[0]['generated_text'][-1]['content'])


print("==================================================================================")

test_question3 = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Alex is Charlie's father. Which one of them was born later?"}
]

response = chatbot(test_question3)
print("ASSISTANT:", response[0]['generated_text'][-1]['content'])

print("==================================================================================")

test_question4 = [
    {"role": "system", "content": "you are a wise and knowledge person."},
    {"role": "user", "content": "How many boxes do I have if I have two boxes with one box inside each?"}
]

response = chatbot(test_question4)
print("ASSISTANT:", response[0]['generated_text'][-1]['content'])

print("==================================================================================")

test_question5 = [{"role": "system", "content": "You are a wise and knowledge person."},
    {"role": "user", "content": "What is 'elbow' spelled backwards?"}
]

response = chatbot(test_question5)
print("ASSISTANT:", response[0]['generated_text'][-1]['content'])




Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASSISTANT: If the Vikings found a smartphone, they would likely be amazed by the device's capabilities. They would probably spend the next few days playing games, browsing the internet, and checking social media. They would also likely be amazed by the smartphone's capabilities, such as being able to take pictures, send emails, and access the internet from almost anywhere. The Vikings might also be tempted to use the smartphone to learn new skills, like how to use it and what features it has. Overall, they would probably be amazed by the smartphone's capabilities and decide to learn more about it.


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASSISTANT: As a wise and knowledgeable person, let's analyze the scenario. 

A mild steel I-beam is a structural element composed of two rectangular cross-sections connected at their ends by a diagonal beam. Given its thickness of ten inches, the beam's length is twenty feet. 

One approach would be to use the formula for the moment of inertia of a rectangular beam, which is I = (1/12)bd^4, where I is the moment of inertia of the beam, b is the beam's width, and d is the beam's depth. However, this formula is quite complex and may not be applicable to our scenario.

A more practical approach is to consider the moment of inertia of the beam itself. The moment of inertia of the entire I-beam structure is the sum of the moment of inertia of each individual beam. 

Considering the beam's geometry, we can use the concept of inelastic strain, which is the change in length of a beam or a structural member caused by a load applied to it in a way that does not create equal and opposite forces. 

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASSISTANT: Alex was Charlie's father.


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASSISTANT: You have 4 boxes.
ASSISTANT: 'Elbow' spelled backwards is 'bowlegs', but in a slang context, it can also be spelled as 'boolegs'.


In [ ]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device

chatbot = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device = device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
test_question5 = [{"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 'generation' spelled backwards?"}
]

response = chatbot(test_question5)
print("ASSISTANT:", response[0]['generated_text'][-1]['content'])

print("==================================================================================")


test_question1 = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Generate a basic blank html with the heading welcome."},
    ]

response = chatbot(test_question1)
print("ASSISTANT:", response[0]['generated_text'][-1]['content'])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASSISTANT: The word "generation" spelled backwards is "nagement."
ASSISTANT: Sure! Here's a simple HTML document that includes a heading "Welcome":

```html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Welcome</title>
</head>
<body>
    <h1>Welcome to my website!</h1>
</body>
</html>
```

This code creates a basic HTML file with a single heading tag `h1` inside an `<body>` section. The heading text is "Welcome to my website!" This is typical for a header in most web pages. You can customize this further as needed by adding more elements or modifying styles as desired.


## Applied Exploration

Choose two instruct models of similar size: https://huggingface.co/models?pipeline_tag=text-generation&sort=trending&search=instruct
  * Link to the model cards for the models you're using and describe each of them

Do one of the following:

1. Go to https://huggingface.co/datasets and find a dataset suitable to use as a benchmark, and compare the performance of the two models. It doesn't have to be a conversational benchmark - it could be a text classification, summarization, math, etc. dataset, as long as you can instruct the model to answer it. And, you don't have to use the whole dataset.
    * link to and describe the dataset
    * describe how you compared the performance (e.g., what metric did you use?)
    * report the results

OR

2. Come up with your own fun benchmark (Taylor Swift trivia, AI Dungeon Master, Joke telling, etc.), generate responses for both models, and have another person rate the answers.
    * Describe what you did
    * report the results
